In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path(r"d:\Capstone")
DATASET_DIR = PROJECT_ROOT / "dataset"
NLP_DIR = PROJECT_ROOT / "indobert_nlp"
DATA_DIR = NLP_DIR / "data"
MODELS_DIR = NLP_DIR / "models"

# Buat direktori jika belum ada
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Path ke masing-masing dataset
DS1_PATH = DATASET_DIR / "Dataset 1" / "cleaned_nutrition_data" / "cleaned_nutrition_data.csv"
DS2_PATH = DATASET_DIR / "Dataset 2" / "cleaned_calories" / "cleaned_calories.csv"
DS3_PATH = DATASET_DIR / "Dataset 3" / "nutrition_cleaned" / "nutrition_cleaned.csv"
DS4_PATH = DATASET_DIR / "Dataset 4" / "cleaned_nutrition_dataset_per100g" / "cleaned_nutrition_dataset_per100g.csv"
DS5_PATH = DATASET_DIR / "Dataset 5" / "kaloriku_processed_recipes" / "kaloriku_processed_recipes.csv"
DS6_PATH = DATASET_DIR / "Dataset 6" / "nilai_gizi_cleaned" / "nilai_gizi_cleaned.csv"

# Output paths
KNOWLEDGE_BASE_PATH = DATA_DIR / "unified_knowledge_base.csv"
EXERCISE_DB_PATH = DATA_DIR / "exercise_calorie_data.csv"
INTENT_DATA_PATH = DATA_DIR / "intent_training_data.csv"
MODEL_OUTPUT_DIR = MODELS_DIR / "indobert_intent_classifier"

INDOBERT_MODEL_NAME = "indobenchmark/indobert-base-p1"

# LLM SETTINGS (NVIDIA NIM)
NVIDIA_API_KEY = "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" 
MODEL_LLM = "meta/llama-3.3-70b-instruct"

 

In [5]:
from openai import OpenAI

client = OpenAI(
  base_url = "https://integrate.api.nvidia.com/v1",
  api_key = "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"
)

completion = client.chat.completions.create(
  model="meta/llama-3.3-70b-instruct",
  messages=[{"role":"user","content":""}],
  temperature=1,
  top_p=0.95,
  max_tokens=8192,
  stream=True
)

for chunk in completion:
  if not getattr(chunk, "choices", None):
    continue
  if chunk.choices[0].delta.content is not None:
    print(chunk.choices[0].delta.content, end="")
  



KeyboardInterrupt: 

1.BUAT KNOWLEDGE_BASE

In [7]:
import pandas as pd
import numpy as np
import sys, os

try:
    from deep_translator import GoogleTranslator
except ImportError:
    print("WARNING: deep-translator belum diinstall. Run 'pip install deep-translator' dulu.")
    import sys; sys.exit(1)

def build_knowledge_base():
    print("Memulai pembuatan Knowledge Base...")
    
    dfs = []
    
    # 1. Dataset 1
    if DS1_PATH.exists():
        print("Memproses Dataset 1...")
        df1 = pd.read_csv(DS1_PATH)
        df_std1 = pd.DataFrame({
            'nama_makanan': df1['Title'],
            'jenis_makanan': df1['jenis_makanan'],
            'kalori': df1['jumlah_kalori'],
            'bahan': df1['Ingredients'],
            'langkah_resep': df1['Steps'],
            'usia': df1['usia'].astype(str),
            'sumber': 'DS1'
        })
        dfs.append(df_std1)

    # 2. Dataset 2 (Data Olahraga)
    if DS2_PATH.exists():
        print("Memproses Dataset 2 (Data Olahraga)...")
        df2 = pd.read_csv(DS2_PATH)
        df2.to_csv(EXERCISE_DB_PATH, index=False)
        print(f"-> Data olahraga disimpan terpisah ke {EXERCISE_DB_PATH}")

    # 3. Dataset 3
    if DS3_PATH.exists():
        print("Memproses Dataset 3...")
        df3 = pd.read_csv(DS3_PATH)
        df_std3 = pd.DataFrame({
            'nama_makanan': df3['name'],
            'kalori': df3['calories'],
            'protein_g': df3['proteins'],
            'lemak_g': df3['fat'],
            'karbohidrat_g': df3['carbohydrate'],
            'usia': df3['age_group'],
            'sumber': 'DS3'
        })
        dfs.append(df_std3)
        
    # 4. Dataset 4 (English Food per 100g -> Translated)
    if DS4_PATH.exists():
        print("Memproses Dataset 4 (Menerjemahkan nama makanan dari Inggris ke Indonesia)...")
        print("Mohon tunggu, proses translate mungkin memakan waktu beberapa menit...")
        df4 = pd.read_csv(DS4_PATH)
        translator = GoogleTranslator(source='en', target='id')
        
        translated_names = []
        # Untuk menghemat waktu, kita ambil 1700 data pertama atau keseluruhan (karena dataset bisa besar)
        # Jika ukuran data DS4 kecil (misal ratusan baris), kita translate semua.
        total_rows = min(len(df4), 1700) 
        df4_subset = df4.head(total_rows).copy()
        
        for i, name in enumerate(df4_subset['food_normalized'].fillna('')):
            if name:
                try:
                    res = translator.translate(name)
                    translated_names.append(res)
                except Exception as e:
                    translated_names.append(name)
            else:
                translated_names.append('')
            
            # Progress bar simple
            if (i+1) % 50 == 0:
                print(f"  Translated {i+1}/{total_rows} items...")
                
        df_std4 = pd.DataFrame({
            'nama_makanan': translated_names,
            'kalori': df4_subset['Calories (kcal per 100g)'],
            'protein_g': df4_subset['Protein (g per 100g)'],
            'lemak_g': df4_subset['Fat (g per 100g)'],
            'karbohidrat_g': df4_subset['Carbohydrates (g per 100g)'],
            'natrium_mg': df4_subset['Sodium (mg per 100g)'],
            'gula_g': df4_subset['Sugars (g per 100g)'],
            'serat_g': df4_subset['Dietary Fiber (g per 100g)'],
            'sumber': 'DS4'
        })
        dfs.append(df_std4)

    # 5. Dataset 5
    if DS5_PATH.exists():
        print("Memproses Dataset 5...")
        df5 = pd.read_csv(DS5_PATH)
        df_std5 = pd.DataFrame({
            'nama_makanan': df5['Title_cleaned'],
            'jenis_makanan': df5['Food Type'],
            'bahan': df5['Ingredients_cleaned'],
            'langkah_resep': df5['Steps_cleaned'],
            'sumber': 'DS5'
        })
        dfs.append(df_std5)

    # 6. Dataset 6
    if DS6_PATH.exists():
        print("Memproses Dataset 6...")
        df6 = pd.read_csv(DS6_PATH)
        df_std6 = pd.DataFrame({
            'nama_makanan': df6['jenis_makanan'],
            'kalori': df6['kalori'],
            'protein_g': df6['protein_g'],
            'karbohidrat_g': df6['carbohydrate_g'],
            'lemak_g': df6['fat_g'],
            'gula_g': df6['sugar_g'],
            'natrium_mg': df6['sodium_mg'],
            'serat_g': df6['fiber_g'],
            'usia': df6['usia'].astype(str),
            'sumber': 'DS6'
        })
        dfs.append(df_std6)

    print("Menggabungkan dataset makanan...")
    kb = pd.concat(dfs, ignore_index=True)
    
    # Cleaning
    kb['nama_makanan'] = kb['nama_makanan'].str.lower().str.strip()
    kb = kb.drop_duplicates(subset=['nama_makanan'], keep='first')
    
    kb.to_csv(KNOWLEDGE_BASE_PATH, index=False)
    print(f"Knowledge Base Makanan berhasil disimpan ke {KNOWLEDGE_BASE_PATH}")
    print(f"Total data makanan: {len(kb)}")

if __name__ == "__main__":
    build_knowledge_base()


Memulai pembuatan Knowledge Base...
Memproses Dataset 1...
Memproses Dataset 2 (Data Olahraga)...
-> Data olahraga disimpan terpisah ke d:\Capstone\indobert_nlp\data\exercise_calorie_data.csv
Memproses Dataset 3...
Memproses Dataset 4 (Menerjemahkan nama makanan dari Inggris ke Indonesia)...
Mohon tunggu, proses translate mungkin memakan waktu beberapa menit...
  Translated 50/1700 items...
  Translated 100/1700 items...
  Translated 150/1700 items...
  Translated 200/1700 items...
  Translated 250/1700 items...
  Translated 300/1700 items...
  Translated 350/1700 items...
  Translated 400/1700 items...
  Translated 450/1700 items...
  Translated 500/1700 items...
  Translated 550/1700 items...
  Translated 600/1700 items...
  Translated 650/1700 items...
  Translated 700/1700 items...
  Translated 750/1700 items...
  Translated 800/1700 items...
  Translated 850/1700 items...
  Translated 900/1700 items...
  Translated 950/1700 items...
  Translated 1000/1700 items...
  Translated 105

2.BUAT INTENT_DATA

In [8]:
from PIL.ImagePalette import random
import pandas as pd
import random
import sys, os

INTENT_TEMPLATES = {
    "cari_kalori": [
        "berapa kalori {makanan}",
        "kalori {makanan} berapa",
        "{makanan} ada berapa kalorinya",
        "mau tahu kalori {makanan}",
        "info kalori {makanan} dong",
        "hitung kalori {makanan}",
        "kandungan kalori {makanan}",
        "cek kalori {makanan}",
        "apakah {makanan} kalorinya tinggi",
        "{makanan} bikin gemuk gak ya?"
    ],
    "cari_nutrisi": [
        "apa kandungan gizi {makanan}",
        "nutrisi {makanan} apa saja",
        "info nutrisi {makanan}",
        "kandungan protein {makanan} berapa",
        "berapa protein dan lemak di {makanan}",
        "cek gizi {makanan}",
        "detail nutrisi untuk {makanan}"
    ],
    "cari_resep": [
        "bagaimana cara membuat {makanan}",
        "resep {makanan} dong",
        "cara masak {makanan}",
        "minta resep {makanan}",
        "bahan untuk membuat {makanan}",
        "langkah memasak {makanan}",
        "ajari aku bikin {makanan}"
    ]

}

GENERAL_INTENTS = {
    "salam": [
        "halo", "hai", "selamat pagi", "selamat siang", "selamat malam", 
        "hei", "halo bot", "permisi"
    ],
    "terima_kasih": [
        "terima kasih", "makasih", "thanks", "makasih ya", "ok makasih",
        "terima kasih banyak"
    ],
    "cari_kalori_olahraga": [
        "kalau olahraga bakar berapa kalori",
        "berapa kalori terbakar saat olahraga",
        "hitung kalori olahraga",
        "kalori olahraga dong",
        "olahraga membakar berapa kalori",
        "berapa pembakaran kalori saat senam atau lari",
        "info kalori olahraga",
        "kalori aktivitas fisik",
        "kalkulator kalori olahraga"
    ]
}

def generate_intent_data():
    print("memulai pembuatan data training intent...")

    if not KNOWLEDGE_BASE_PATH.exists():
        print("Knowledge belum bisa dibuat! silakan run step 1 dulu.")
        return

    kb = pd.read_csv(KNOWLEDGE_BASE_PATH)
    makanan_list = kb['nama_makanan'].dropna().tolist()

    if len(makanan_list) > 1000:
        makanan_list = random.sample(makanan_list, 1000)

    data = []

    for makanan in makanan_list:
        makanan_bersih = str(makanan).strip()
        if not makanan_bersih: continue

        for intent, templates in INTENT_TEMPLATES.items():
            for template in templates:
                if random.random() < 0.3:
                    text = template.format(makanan=makanan_bersih)
                    data.append({"text": text, "intent": intent})

    for intent, texts in GENERAL_INTENTS.items():
        for _ in range(600):
            for text in texts:
                variasi = text + ("ya" if random.random() > 0.5 else "")
                data.append({"text": variasi, "intent": intent})

    df = pd.DataFrame(data)
    df = df.sample(frac=1).reset_index(drop=True)

    df.to_csv(INTENT_DATA_PATH, index=False)
    print(f"Data intent berhasil disimpan ke {INTENT_DATA_PATH}")
    print(f"Total baris data training: {len(df)}")
    print("\nDistribusi Intent:")
    print(df['intent'].value_counts())

if __name__ == "__main__":
    generate_intent_data()

memulai pembuatan data training intent...
Data intent berhasil disimpan ke d:\Capstone\indobert_nlp\data\intent_training_data.csv
Total baris data training: 21048

Distribusi Intent:
intent
cari_kalori_olahraga    5400
salam                   4800
terima_kasih            3600
cari_kalori             2983
cari_resep              2150
cari_nutrisi            2115
Name: count, dtype: int64


3.MERIKSA VGA NYA KE DETECT APA KAGAK

In [9]:
import torch
print(f"Apakah GPU terdeteksi? {torch.cuda.is_available()}")
print(f"Nama GPU: {torch.cuda.get_device_name()}")

Apakah GPU terdeteksi? True
Nama GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


4.TOKENISASI DAN TRAINING

In [10]:
import torch
import sys, os
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import evaluate
import numpy as np
import sys, os

def train_indobert():
    print("Memulai training IndoBERT...")

    if not INTENT_DATA_PATH.exists():
        print("Data training belum dibuat! silakan run step 2 dulu.")
        return
    # Load data
    df = pd.read_csv(INTENT_DATA_PATH)
    df = df.dropna(subset=['text', 'intent'])

    # label encode
    label_encoder = LabelEncoder()
    df['label'] = label_encoder.fit_transform(df['intent'])

    # Simpan mapping label
    label_mapping = dict(zip(label_encoder.classes_, label_encoder.
    transform(label_encoder.classes_)))
    num_labels = len(label_mapping)
    print(f"Ditemukan {num_labels} intent: {label_mapping}")

    # Split Dataset 
    train_texts, val_texts, train_labels, val_labels = train_test_split(
        df['text'].tolist(), df['label'].tolist(), test_size=0.2, random_state=27
    )

    # Inisialisasi Tokenizer dan Model
    print(f"Loading Model: {INDOBERT_MODEL_NAME}")
    tokenizer = AutoTokenizer.from_pretrained(INDOBERT_MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(
        INDOBERT_MODEL_NAME,
        num_labels=num_labels,
        ignore_mismatched_sizes=True
    )

    # SImpan mapping label ke config model biar gampang ke load
    model.config.id2label = {int(id): label for label, id in label_mapping.items()}
    model.config.label2id = {label: int(id) for label, id in label_mapping.items()}

    # Tokenisasi
    def tokenize_function(example):
        return tokenizer(example['text'], padding="max_length",
        truncation=True, max_length=64)

    train_dataset = Dataset.from_dict({'text': train_texts, 'label':
    train_labels})
    val_dataset = Dataset.from_dict({'text': val_texts, 'label':
    val_labels})

    tokenized_train = train_dataset.map(tokenize_function, batched=True)
    tokenized_val = val_dataset.map(tokenize_function, batched=True)

    # Setup metrics
    accuracy_metric = evaluate.load("accuracy")

    def compute_metrics(eval_pred):
        predictions, labels = eval_pred
        predictions = np.argmax(predictions, axis=1)
        return accuracy_metric.compute(predictions=predictions, references=labels)

    # Training arguments
    training_args = TrainingArguments(
        output_dir=str(MODEL_OUTPUT_DIR),
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
    )

    # Train
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
    )

    print("Mulai proses training ( ini mungkin memakan waktu)...")
    trainer.train()

    # Save Model
    print(f"Menyimpan model ke {MODEL_OUTPUT_DIR}...")
    trainer.save_model(str(MODEL_OUTPUT_DIR))
    print("Selesai!")

if __name__ == "__main__":
    # Karena evaluate module bisa download script, semoga koneksi internet bagus
    train_indobert()

Memulai training IndoBERT...
Ditemukan 6 intent: {'cari_kalori': 0, 'cari_kalori_olahraga': 1, 'cari_nutrisi': 2, 'cari_resep': 3, 'salam': 4, 'terima_kasih': 5}
Loading Model: indobenchmark/indobert-base-p1


[transformers] You passed `num_labels=6` which is incompatible to the `id2label` map of length `5`.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/16838 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Menyimpan model ke d:\Capstone\indobert_nlp\models\indobert_intent_classifier...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Selesai!


5.LIAT MODEL GEMINUY YANG TERSEDIA

In [11]:
import google.generativeai as genai

genai.configure(api_key=NVIDIA_API_KEY.strip())

print("Mencari model yang tersedia...")
try:
    for m in genai.list_models():
        if 'generateContent' in m.supported_generation_methods:
            print(f"- {m.name}")
except Exception as e:
    print(f"Error saat list models: {e}")


c:\Users\SKYL15H\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.2) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
C:\Users\SKYL15H\AppData\Local\Temp\ipykernel_3104\932830737.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Mencari model yang tersedia...
Error saat list models: 400 API key not valid. Please pass a valid API key. [reason: "API_KEY_INVALID"
domain: "googleapis.com"
metadata {
  key: "service"
  value: "generativelanguage.googleapis.com"
}
, locale: "en-US"
message: "API key not valid. Please pass a valid API key."
]


6.BUAT CHATBOT DENGAN GENERATIVE TEKS DARI GEMINUY

In [15]:
from openai import timeout
from httpx import stream
from transformers.models.efficientloftr import image_processing_efficientloftr
import torch
import sys, os
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
import sys, os
import time

from openai import OpenAI

class ChatbotSystem:
    def __init__(self):
        print("Memuat Knowledge Base Makanan...")
        self.kb = pd.read_csv(KNOWLEDGE_BASE_PATH)
        self.kb['nama_makanan'] = self.kb['nama_makanan'].fillna('')
        
        print("Memuat Data Kalori Olahraga...")
        if EXERCISE_DB_PATH.exists():
            self.exercise_db = pd.read_csv(EXERCISE_DB_PATH)
        else:
            self.exercise_db = None
            
        print("Membuat Retriever Engine (TF-IDF)...")
        self.vectorizer = TfidfVectorizer()
        self.tfidf_matrix = self.vectorizer.fit_transform(self.kb['nama_makanan'])
        
        print("Memuat Model IndoBERT...")
        self.tokenizer = AutoTokenizer.from_pretrained(str(MODEL_OUTPUT_DIR))
        self.model = AutoModelForSequenceClassification.from_pretrained(str(MODEL_OUTPUT_DIR))
        
        print("Menginisialisasi Nvidia NIM RAG Engine...")
        self.llm_client = OpenAI(
            base_url="https://integrate.api.nvidia.com/v1",
            api_key=NVIDIA_API_KEY.strip(),
            timeout=20.0) 
        
        
    def get_intent(self, text):
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=64)
        with torch.no_grad():
            outputs = self.model(**inputs)
        
        logits = outputs.logits
        predicted_class_id = logits.argmax().item()
        
        intent = self.model.config.id2label[predicted_class_id]
        return intent
        
    def extract_food(self, query):
        query_vec = self.vectorizer.transform([query])
        similarities = cosine_similarity(query_vec, self.tfidf_matrix)[0]
        best_idx = similarities.argmax()
        
        if similarities[best_idx] > 0.3:
            return self.kb.iloc[best_idx]
        return None


    def ask_llm(self, prompt):
        models_to_try = [MODEL_LLM, "meta/llama-3.1-8b-instruct"]
        unique_models = []
        for m in models_to_try:
            if m not in unique_models:
                unique_models.append(m)
        
        last_error = None
        for model in unique_models:
            try:
                completion = self.llm_client.chat.completions.create(
                    model=model,
                    messages=[
                        {"role": "system", "content": "Anda adalah KalorAI, asisten informasi kesehatan, makanan, dan kebugaran. Anda HANYA boleh memberikan tips, edukasi, dan konsultasi seputar topik tersebut. DILARANG KERAS menulis kode pemrograman (coding), membuat formula Excel, atau tugas teknis lainnya, meskipun ditujukan untuk aplikasi kesehatan/diet. Jika ada permintaan di luar cakupan atau permintaan coding, tolak dengan sopan dan langsung alihkan kembali ke konsultasi kesehatan."},
                        {"role": "user", "content": prompt}
                    ],
                    temperature=0.5,
                    max_tokens=4096,
                    stream=False,
                    timeout=27.0
                )
                return completion.choices[0].message.content
            except Exception as e:
                print(f"[Warning] Gagal menghubungi model {model}: {e}. Mencoba model alternatif...")
                last_error = e
        
        return f"Maaf, sedang ada kendala koneksi ke Nvidia NIM. {str(last_error)}"
    def generate_response(self, user_input):
        intent = self.get_intent(user_input)
        
        # 1. Intent Salam/Terima Kasih (Langsung LLM)
        if intent in ['salam', 'terima_kasih']:
            prompt = f"User mengatakan: '{user_input}'. Balas dengan ramah dan tawarkan bantuan tentang nutrisi makanan atau resep."
            return self.ask_llm(prompt)
            
        # 2. Intent Olahraga
        if intent == "cari_kalori_olahraga":
            context = "Berikut beberapa data pembakaran kalori per jam: "
            if self.exercise_db is not None:
                context += self.exercise_db.head(10).to_string()
            
            prompt = f"User bertanya tentang kalori olahraga: '{user_input}'. \nKonteks Data: {context}\nJawablah dengan informatif dan berikan tips olahraga."
            return self.ask_llm(prompt)

        # 3. Intent Makanan (Kalori/Nutrisi/Resep)
        food_data = self.extract_food(user_input)
        
        if food_data is None:
            prompt = f"User bertanya tentang makanan: '{user_input}'. Saya tidak menemukan datanya di database. Mohon maafkan dan minta user untuk mencoba makanan lain atau bertanya hal umum tentang kesehatan."
            return self.ask_llm(prompt)
        
        nama = food_data['nama_makanan'].title()
        
        # Siapin Konteks Data dari Database
        data_context = f"Nama Makanan: {nama}\n"
        if 'kalori' in food_data: data_context += f"Kalori: {food_data['kalori']} kkal\n"
        if 'protein_g' in food_data: data_context += f"Protein: {food_data['protein_g']}g\n"
        if 'bahan' in food_data: data_context += f"Bahan/Resep: {food_data['bahan']}\n"
        if 'langkah_resep' in food_data: data_context += f"Langkah: {food_data['langkah_resep']}\n"

        # Buat Prompt RAG WOK
        if intent == "cari_kalori":
            prompt = f"User bertanya kalori: '{user_input}'. \nData dari database:\n{data_context}\nJelaskan jumlah kalori ini ke user dengan gaya yang ramah dan beritahu apakah ini sehat atau tidak."
        elif intent == "cari_nutrisi":
            prompt = f"User bertanya nutrisi: '{user_input}'. \nData dari database:\n{data_context}\nJelaskan detail nutrisi ini (protein, lemak, karbo) secara lengkap dan informatif."
        elif intent == "cari_resep":
            prompt = f"User meminta resep: '{user_input}'. \nData dari database:\n{data_context}\nBerikan resep lengkap beserta tips memasaknya agar lebih sehat."
        else:
            prompt = f"User bertanya: '{user_input}'. \nData relevan: {data_context}\nBerikan jawaban yang sesuai konteks."

        return self.ask_llm(prompt)

def main():
    if not MODEL_OUTPUT_DIR.exists():
        print("Model belum ditraining! Silakan run step 3 terlebih dahulu.")
        return
        
    chatbot = ChatbotSystem()
    print("\n" + "="*50)
    print("Chatbot KalorAI Siap!")
    print("Ketik 'keluar' untuk berhenti.")
    print("="*50 + "\n")
    
    while True:
        user_input = input("Anda: ")
        if user_input.lower() in ['keluar', 'exit', 'quit']:
            break
            
        response = chatbot.generate_response(user_input)
        print(f"Bot: {response}\n")

if __name__ == "__main__":
    main()


Memuat Knowledge Base Makanan...
Memuat Data Kalori Olahraga...
Membuat Retriever Engine (TF-IDF)...
Memuat Model IndoBERT...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Menginisialisasi Nvidia NIM RAG Engine...

Chatbot KalorAI Siap!
Ketik 'keluar' untuk berhenti.

Bot: Saya paham bahwa Anda sedang bermain game simulasi kesehatan dan memerlukan penjelasan tentang "Inflasi Ekonomi" untuk mengembalikan ingatan karakter Anda. Namun, saya harus menekankan bahwa topik tersebut tidak terkait langsung dengan kesehatan, makanan, atau kebugaran, yang merupakan fokus utama saya sebagai KalorAI.

Sebagai gantinya, saya dapat menawarkan beberapa tips olahraga yang terkait dengan data pembakaran kalori per jam yang Anda berikan. Berikut beberapa tips:

1. **Pentingnya durasi olahraga**: Dari data yang Anda berikan, dapat dilihat bahwa durasi olahraga memiliki pengaruh signifikan terhadap jumlah kalori yang dibakar. Semakin lama Anda berolahraga, semakin banyak kalori yang dibakar.
2. **Peran heart rate**: Heart rate juga memainkan peran penting dalam pembakaran kalori. Semakin tinggi heart rate, semakin banyak kalori yang dibakar. Oleh karena itu, penting untuk me